<a href="https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krithi-ks/flyrankAI-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ML-08 Setup
# Reconnect to the warehouse: March 2026 = feature window, April 2026 = outcome window.

%pip -q install duckdb huggingface_hub scikit-learn

import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found. Add your READ Hugging Face token to Colab Secrets as HF_TOKEN.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FEATURE_MONTH = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet'"
    ")"
)
OUTCOME_MONTH = (
    "read_parquet("
    "'hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet'"
    ")"
)

print("Feature window: March 2026")
print("Outcome window: April 2026 (next month — still short of the sealed June test month)")

# ---------------------------------------------------------
# 1. Rebuild the five verified features (same as ML-04 Section 3),
#    plus a GA4-availability flag so we don't fillna(0) blind.
# ---------------------------------------------------------
five_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_mar,
        SUM(gsc_clicks) AS clicks_mar,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS avg_position_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE NULL END) AS sessions_mar,
        COUNT(DISTINCT report_date) AS observed_days_mar,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_mar,
        MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS has_gsc_mar
    FROM {FEATURE_MONTH}
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Feature rows (March): {len(five_features):,}")

# ---------------------------------------------------------
# 2. Rebuild the ML-07 baseline rule on the same March aggregation.
# ---------------------------------------------------------
baseline_df = five_features.rename(columns={
    "impressions_mar": "impressions",
    "clicks_mar": "clicks",
    "avg_position_mar": "avg_position",
})[["client_hash_id", "content_hash_id", "impressions", "clicks", "avg_position"]].copy()

baseline_df["visible"] = (baseline_df["impressions"] >= 500).astype(int)
baseline_df["weak_position"] = (baseline_df["avg_position"] > 20).fillna(False).astype(int)
baseline_df["has_clicks"] = (baseline_df["clicks"] > 0).fillna(False).astype(int)
baseline_df["baseline_score"] = (
    baseline_df["visible"] * 2 + baseline_df["weak_position"] * 2 + baseline_df["has_clicks"]
)
baseline_df["reason_code"] = np.select(
    [
        (baseline_df["visible"] == 1) & (baseline_df["weak_position"] == 1),
        (baseline_df["visible"] == 1) & (baseline_df["has_clicks"] == 1),
    ],
    ["visible_but_weak_position", "visible_and_clickable"],
    default="review_other",
)

# ---------------------------------------------------------
# 3. Build the future outcome from April 2026.
#    Only GSC-available rows count as real April observations —
#    same "don't treat unavailable as zero" rule from ML-04.
# ---------------------------------------------------------
april_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS future_gsc_impressions
    FROM {OUTCOME_MONTH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f"Outcome rows (April, GSC-available): {len(april_outcome):,}")

# ---------------------------------------------------------
# 4. Assemble the modeling frame.
#    Minimum-volume filter (impressions_mar >= 100) controls noise,
#    matching the smallest meaningful bucket from ML-06.
#    Inner join on has_gsc_mar + April presence: only content items
#    observed in BOTH windows get a label — no fabricated zeros.
# ---------------------------------------------------------
df = five_features.merge(baseline_df.drop(columns=["impressions", "clicks", "avg_position"]),
                          on=["client_hash_id", "content_hash_id"])
df = df.merge(april_outcome, on=["client_hash_id", "content_hash_id"], how="inner")

df = df[(df["has_gsc_mar"] == 1) & (df["impressions_mar"] >= 100)].copy()

df["pct_change"] = (df["future_gsc_impressions"] - df["impressions_mar"]) / df["impressions_mar"]
df["future_outcome"] = (df["pct_change"] <= -0.30).astype(int)

print(f"\nModeling frame rows: {len(df):,}")
print(f"Base rate of future_outcome (decline): {df['future_outcome'].mean():.3f}")

# ---------------------------------------------------------
# 5. Leakage assertion — same excluded fields from ML-04.
# ---------------------------------------------------------
excluded_fields = {"trend_direction", "trend_pct", "health_score", "priority_score", "action_type", "refresh_tier"}
present = excluded_fields.intersection(set(df.columns))
assert not present, f"Leakage detected: {present}"
print("PASS: no excluded product-decision or trend fields in the modeling frame.")

Feature window: March 2026
Outcome window: April 2026 (next month — still short of the sealed June test month)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows (March): 331,437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Outcome rows (April, GSC-available): 194,760

Modeling frame rows: 100,893
Base rate of future_outcome (decline): 0.430
PASS: no excluded product-decision or trend fields in the modeling frame.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

future_outcome is a yes/no observed label (did April impressions drop ≥30% vs March). Per the training playbook, that shape starts with Logistic Regression — a model I can read a coefficient table from and explain to a non-technical reviewer — then Random Forest, to see whether the extra complexity earns its keep over a linear model. I'm not reaching for gradient boosting or anything heavier: with five features and one month of outcome data, a forest that beats logistic regression by a wide, stable margin is already a strong finding: If it doesn't, that's also a valid, reportable result — not a failure.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Label shape: binary (future_outcome)")
print(f"Base rate: {df['future_outcome'].mean():.3f}  (n={len(df):,})")
print("Method order: Logistic Regression (readable) -> Random Forest (stronger, if it earns it)")

Label shape: binary (future_outcome)
Base rate: 0.430  (n=100,893)
Method order: Logistic Regression (readable) -> Random Forest (stronger, if it earns it)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Multiple content items belong to the same client, and clients likely share site-wide patterns (template, niche, seasonality) a model could memorize instead of learning generalizable signal. A row-level random split would let the same client appear in both train and test, which is exactly the leakage risk called out in the lane guide's validation checklist. So I split by client_hash_id: every content item from a given client lands entirely in train or entirely in test. This is the same discipline ML-04/ML-07's design already assumed (client as the grouping key), just enforced at model-training time.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


FEATURE_COLS = [
    "impressions_mar", "clicks_mar", "avg_position_mar",
    "sessions_mar", "observed_days_mar", "has_ga4_mar",
]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

print(f"Train rows: {len(train_df):,}  |  Test rows: {len(test_df):,}")
print(f"Train clients: {len(train_clients)}  |  Test clients: {len(test_clients)}")
print(f"Client overlap between train and test: {len(train_clients & test_clients)}")  # must be 0
assert len(train_clients & test_clients) == 0, "Client leakage across the split"
print("PASS: no client appears in both train and test.")

Train rows: 84,384  |  Test rows: 16,509
Train clients: 32  |  Test clients: 11
Client overlap between train and test: 0
PASS: no client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same modeling frame, same client-grouped split, same test set for the baseline rule and both models. I score everything — the baseline's baseline_score, and each model's predicted probability of future_outcome — as a ranking on the held-out test set, then compare ROC AUC, average precision, and Precision@20/@50 (the same top-K sizes ML-07's manual review used). If a model beats the rule at one K but loses at another, I report both, not just the flattering one.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    top_k = y_true.values[order][:k]
    return top_k.mean()

X_train, y_train = train_df[FEATURE_COLS], train_df["future_outcome"]
X_test, y_test = test_df[FEATURE_COLS], test_df["future_outcome"]

# --- Logistic Regression: impute + scale ---
logreg_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
logreg_pipe.fit(X_train, y_train)
logreg_scores = logreg_pipe.predict_proba(X_test)[:, 1]

# --- Random Forest: impute only, no scaling needed ---
rf_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20,
        random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
rf_pipe.fit(X_train, y_train)
rf_scores = rf_pipe.predict_proba(X_test)[:, 1]

# --- Baseline: the ML-07 rule score, unchanged ---
baseline_scores = test_df["baseline_score"].values

results = []
for name, scores in [
    ("baseline rule (ML-07)", baseline_scores),
    ("logistic regression", logreg_scores),
    ("random forest", rf_scores),
]:
    results.append({
        "method": name,
        "roc_auc": roc_auc_score(y_test, scores),
        "avg_precision": average_precision_score(y_test, scores),
        "precision@20": precision_at_k(y_test, scores, 20),
        "precision@50": precision_at_k(y_test, scores, 50),
    })

comparison_table = pd.DataFrame(results)
comparison_table["base_rate"] = y_test.mean()
display(comparison_table.round(3))

,method,roc_auc,avg_precision,precision@20,precision@50,base_rate
0,baseline rule (ML-07),0.432,0.285,0.10,0.16,0.318
1,logistic regression,0.552,0.359,0.55,0.54,0.318
2,random forest,0.574,0.371,0.65,0.54,0.318


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

A metric table isn't the finding — reading the errors is. I check which features the random forest leans on (and sanity-check the top feature makes sense rather than looking suspiciously perfect), where the model is most wrong by reason code, and I pull three concrete wrong cases to explain in plain words.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Feature importances
importances = pd.Series(
    rf_pipe.named_steps["clf"].feature_importances_, index=FEATURE_COLS
).sort_values(ascending=False)
print("Random Forest feature importances:")
display(importances)

# Error breakdown by baseline reason_code — where is the model most wrong?
test_df = test_df.copy()
test_df["rf_prob"] = rf_scores
test_df["rf_pred"] = (test_df["rf_prob"] >= 0.5).astype(int)
test_df["correct"] = (test_df["rf_pred"] == test_df["future_outcome"]).astype(int)

error_by_reason = test_df.groupby("reason_code")["correct"].agg(["mean", "count"])
error_by_reason.columns = ["accuracy", "n"]
print("\nAccuracy by ML-07 reason code:")
display(error_by_reason)

# Three concrete wrong cases
false_negatives = test_df[(test_df["future_outcome"] == 1) & (test_df["rf_pred"] == 0)] \
    .sort_values("rf_prob").head(2)
false_positives = test_df[(test_df["future_outcome"] == 0) & (test_df["rf_pred"] == 1)] \
    .sort_values("rf_prob", ascending=False).head(1)

wrong_cases = pd.concat([false_negatives, false_positives])[
    ["client_hash_id", "content_hash_id", "impressions_mar", "avg_position_mar",
     "future_gsc_impressions", "pct_change", "rf_prob", "future_outcome"]
]
print("\nThree concrete wrong cases:")
display(wrong_cases)

Random Forest feature importances:


,0
clicks_mar,0.385722
observed_days_mar,0.262058
impressions_mar,0.106811
avg_position_mar,0.105953
has_ga4_mar,0.086944
sessions_mar,0.052513



Accuracy by ML-07 reason code:


,accuracy,n
reason_code,,
review_other,0.478442,8141
visible_and_clickable,0.695048,6624
visible_but_weak_position,0.564794,1744



Three concrete wrong cases:


,client_hash_id,content_hash_id,impressions_mar,avg_position_mar,future_gsc_impressions,pct_change,rf_prob,future_outcome
81690,client_0fa64a184f18a4a0,content_5490363da423c29f,1734.0,2.148743,1020.0,-0.411765,0.053932,1
169681,client_2094c6eb080311d5,content_ee6780540faea277,1227.0,5.243433,441.0,-0.640587,0.059122,1
41027,client_e547b89c05043229,content_f9e04ee043a37ad6,2962.0,1.386007,2819.0,-0.048278,0.630095,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.